In [ ]:
# %%
import math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("results/figs", exist_ok=True)
plt.rcParams["figure.figsize"] = (9, 5)
QUICK = False
N_SEEDS = 40 if QUICK else 200

# ---- same world as the notebook (do not change, or results will not match) ----
_layout = np.random.default_rng(7)
N_NODES = 12
XY = _layout.uniform(0, 900, size=(N_NODES, 2))
SPEED = 130.0
T = np.ceil(np.linalg.norm(XY[:, None] - XY[None], axis=2) / SPEED).astype(int)
N_BINS = N_NODES - 1
BASE_RATE = _layout.uniform(20, 48, size=N_BINS)
DAY_MIN, START_HOUR = 720, 8


def demand_curve(h):
    return (0.25 + 1.2 * math.exp(-((h - 12.5) / 1.2) ** 2)
            + 0.6 * math.exp(-((h - 10.0) / 0.6) ** 2)
            + 0.6 * math.exp(-((h - 15.0) / 0.7) ** 2))


CURVE = np.array([demand_curve(START_HOUR + t / 60) for t in range(DAY_MIN)])


def fill_paths(seed, scale=1.0):
    """Fill level (%) of every bin every minute with NO robots: shape (DAY_MIN, N_BINS).
    Uses the same RNG draw order as simulate() so seed k is the same day."""
    rng = np.random.default_rng(seed)
    day_mult = rng.lognormal(0, 0.2, N_BINS)
    noise = np.clip(1 + 0.3 * rng.standard_normal((DAY_MIN, N_BINS)), 0, None)
    start = rng.uniform(0, 50, N_BINS)
    inc = BASE_RATE * day_mult * CURVE[:, None] * scale * noise / 60
    return np.minimum(start + np.cumsum(inc, axis=0), 100.0), day_mult


PATHS = [fill_paths(s)[0] for s in range(N_SEEDS)]
DAYMULT = np.array([fill_paths(s)[1] for s in range(N_SEEDS)])
print(f"{N_SEEDS} simulated days, {N_BINS} bins")

# %% [markdown]
# ## B. Response window and alert threshold  (notify.py / supervisor.py)
# If nobody collects a bin, how long between crossing a threshold and hitting 100%?
# That is the time budget a dispatcher has. It also tells us where to set the alert.

# %%
rows = []
for s, P in enumerate(PATHS):
    for i in range(N_BINS):
        for thr in (60, 70, 80, 90):
            hit = np.flatnonzero(P[:, i] >= thr)
            if hit.size == 0:
                continue
            t0 = hit[0]
            full = np.flatnonzero(P[:, i] >= 100)
            lead = (full[0] - t0) if full.size else np.nan   # NaN = never overflowed that day
            rows.append((s, i, thr, t0, lead))
win = pd.DataFrame(rows, columns=["seed", "bin", "thr", "t0", "lead_min"])
win["hour"] = START_HOUR + win.t0 // 60
win.to_csv("results/response_windows.csv", index=False)

summ = (win.groupby("thr").lead_min
        .agg(share_overflow_if_ignored=lambda s: s.notna().mean(),
             median="median", p10=lambda s: s.quantile(0.1), p90=lambda s: s.quantile(0.9),
             share_under_15=lambda s: (s <= 15).mean()))
print("\nMinutes from crossing a threshold to overflow if nobody comes:")
print(summ.round(2).to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
for thr in (60, 70, 80, 90):
    d = win[(win.thr == thr) & win.lead_min.notna()].lead_min.sort_values()
    ax[0].plot(d, np.arange(1, len(d) + 1) / len(d), label=f"{thr}% threshold")
ax[0].axvline(15, ls="--", color="gray")
ax[0].set(xlabel="minutes until 100% if ignored", ylabel="ECDF", title="Time budget after an alert")
ax[0].legend()
d80 = win[(win.thr == 80) & win.lead_min.notna()]
g = d80.groupby("hour").lead_min
ax[1].plot(g.median().index, g.median().values, marker="o", label="median")
ax[1].fill_between(g.median().index, g.quantile(0.1).values, g.quantile(0.9).values, alpha=0.2, label="10-90%")
ax[1].set(xlabel="hour bin crossed 80%", ylabel="minutes to overflow",
          title="Response window shrinks at the lunch peak")
ax[1].legend(); plt.tight_layout()
plt.savefig("results/figs/B_response_window.png", dpi=130); plt.show()
